# OOD stress test — anisotropic / heavy-tailed embeddings — [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahb-sjsu/turboquant-pro/blob/master/notebooks/claims/04_ood_anisotropic.ipynb)

**Evidence-ladder rung:** Track 1 · L1 (Colab). Robustness check requested in review: public sets (GloVe, LaBSE) are relatively well-behaved, but heavily fine-tuned domain encoders (biomedical, legal) can have **extreme spatial anisotropy** and non-Gaussian rotated tails.

We synthesize a pathological corpus — a few dominant directions (rank-deficient core), a **power-law eigenvalue spectrum**, Student-t heavy tails, and a random rotation — and run the same canonical ladder against exact ground truth. The question: does PCA+TQ degrade gracefully, and does the honest dataset-dependence story still hold?

## 1. Install

In [ ]:
!pip install -q turboquant-pro faiss-cpu numpy pandas

## 2. Configure

In [ ]:
N          = 60_000   # corpus size
QUERIES    = 1000
DIM        = 256
ANISO      = 2.0      # power-law exponent for the eigenvalue decay (larger = more anisotropic)
TAIL_DF    = 3.0      # Student-t degrees of freedom (smaller = heavier tails)
OUT_DIM    = 128
BITS       = 3
OVERSAMPLE = 5
SEED       = 0
THREADS    = 8

## 3. Harness (verified, embedded)

In [ ]:
# === embedded harness (verbatim from benchmarks/canonical_embedding.py) ===
# Single source of truth for the method ladder; identical to the CI-tested file.
#!/usr/bin/env python3
"""Canonical embedding-compression benchmark harness (Review-1 deliverable).

ONE table, ALL methods, IDENTICAL rerank protocol — so the headline claim
("beats RaBitQ on recall, ties OPQ at scale, builds faster") is reproducible
in a single call. Used by notebooks/claims/00_canonical_sota_embedding.ipynb
(auto-downloads public GloVe) and callable directly on any corpus.

Methods (all at a matched ~out_dim*bits byte budget where applicable):
  fp32-flat  PQ  OPQ  IVFPQ  RaBitQ  PCA-only  TQ-only  PCA+TQ  ADCIndex

Every ANN method is measured twice: single-stage, and +rerank with the SAME
oversample factor (candidates = 10 * oversample), reranked by exact fp32 cosine
on the retained originals. bytes/vector is computed ANALYTICALLY (out_dim*bits/8)
to keep this harness self-contained and library-agnostic. (Note: the library's
PCAMatryoshkaPipeline.estimate_storage() was dimension-agnostic before v1.4.1
and now tracks the real config; we still compute analytically here.)

    from canonical_embedding import run_canonical, to_markdown
    rows = run_canonical(C, Q, gt, out_dim=64, bits=3, oversample=5)
    print(to_markdown(rows))
"""

from __future__ import annotations

import time
from collections.abc import Sequence

import numpy as np


# ---------------------------------------------------------------- primitives
def normalize(X: np.ndarray) -> np.ndarray:
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-30)


def exact_topk(Q: np.ndarray, X: np.ndarray, k: int) -> np.ndarray:
    """Exact cosine top-k ground truth (for corpora without provided neighbors)."""
    out = np.zeros((len(Q), k), dtype=np.int64)
    for s in range(0, len(Q), 256):
        sims = Q[s : s + 256] @ X.T
        idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for r in range(idx.shape[0]):
            idx[r] = idx[r][np.argsort(-sims[r, idx[r]])]
        out[s : s + 256] = idx
    return out


def recall(gt: np.ndarray, ap: np.ndarray, k: int) -> float:
    return float(
        np.mean([len(set(gt[i, :k]) & set(ap[i, :k])) / k for i in range(len(gt))])
    )


def _rerank(cand: np.ndarray, Q: np.ndarray, C: np.ndarray, k: int = 10) -> np.ndarray:
    """Two-stage: re-rank candidate ids by exact fp32 cosine on the originals."""
    rr = np.full((len(Q), k), -1, dtype=np.int64)
    for i in range(len(Q)):
        c = cand[i][cand[i] >= 0]
        if len(c) == 0:
            continue
        s = C[c] @ Q[i]
        top = c[np.argsort(-s)[:k]]
        rr[i, : len(top)] = top
    return rr


def _divisor_m(target_m: int, dim: int) -> int:
    m = min(max(1, target_m), dim)
    while dim % m != 0:
        m -= 1
    return m


# ---------------------------------------------------------------- main driver
def run_canonical(
    C: np.ndarray,
    Q: np.ndarray,
    gt: np.ndarray,
    out_dim: int = 64,
    bits: int = 3,
    oversample: int = 5,
    methods: Sequence[str] | None = None,
    threads: int = 8,
    reps: int = 2,
    train_cap: int = 200_000,
) -> list[dict]:
    """Run the canonical method ladder. C/Q assumed L2-normalized. gt = top-k ids.

    Returns a list of row dicts with identical schema across methods so the
    caller can render one table. `oversample` is shared by EVERY +rerank row.
    """
    import faiss

    faiss.omp_set_num_threads(threads)
    from turboquant_pro import ADCIndex, PCAMatryoshka

    N, dim = C.shape
    nq = len(Q)
    kcand = 10 * oversample
    train = C[: min(N, train_cap)]
    all_methods = ["flat", "pq", "opq", "ivfpq", "rabitq", "pca", "tq", "pca_tq", "adc"]
    M = list(methods) if methods else all_methods
    rows: list[dict] = []

    def bench(fn) -> float:
        best = 1e30
        for _ in range(reps):
            t = time.perf_counter()
            fn()
            best = min(best, time.perf_counter() - t)
        return nq / best

    def add(method, bpv, build_s, qps1, qps_rr, r10_1, r10_rr, r100, note=""):
        rows.append(
            dict(
                method=method,
                n=N,
                dim=dim,
                bytes_per_vec=round(float(bpv), 1),
                compression_x=round(dim * 4 / float(bpv), 1),
                build_s=round(float(build_s), 3),
                qps_1stage=None if qps1 is None else round(float(qps1), 1),
                qps_rerank=None if qps_rr is None else round(float(qps_rr), 1),
                recall_at_10=None if r10_1 is None else round(float(r10_1), 4),
                recall_at_10_rerank=None if r10_rr is None else round(float(r10_rr), 4),
                recall_at_100=None if r100 is None else round(float(r100), 4),
                ram_mb=round(float(bpv) * N / 1e6, 1),
                note=note,
            )
        )
        print("  ", rows[-1], flush=True)

    # -- fp32-flat exact baseline -------------------------------------------
    if "flat" in M:
        t = time.perf_counter()
        flat = faiss.IndexFlatIP(dim)
        flat.add(C)
        bt = time.perf_counter() - t
        _, nn = flat.search(Q, 100)
        qps = bench(lambda: flat.search(Q, 10))
        add(
            "fp32-flat",
            dim * 4,
            bt,
            qps,
            None,
            recall(gt, nn, 10),
            None,
            recall(gt, nn, 100),
            "exact baseline",
        )

    # -- PQ / OPQ at matched byte budget ------------------------------------
    for fac, tag in (("PQ", "pq"), ("OPQ", "opq")):
        if tag not in M:
            continue
        m = _divisor_m((out_dim * bits) // 8, dim)
        try:
            spec = f"PQ{m}x8" if fac == "PQ" else f"OPQ{m},PQ{m}"
            t = time.perf_counter()
            index = faiss.index_factory(dim, spec, faiss.METRIC_INNER_PRODUCT)
            index.train(train)
            index.add(C)
            bt = time.perf_counter() - t
            _, nn = index.search(Q, 100)
            _, cand = index.search(Q, kcand)
            rr = _rerank(cand, Q, C)
            q1 = bench(lambda index=index: index.search(Q, 10))
            qr = bench(lambda index=index: index.search(Q, kcand))
            add(
                f"faiss-{fac}(m={m})",
                m,
                bt,
                q1,
                qr,
                recall(gt, nn, 10),
                recall(gt, rr, 10),
                recall(gt, nn, 100),
                f"~{bits}-bit budget; +rerank x{oversample}",
            )
        except Exception as e:  # noqa: BLE001
            print(f"  {fac} failed: {e}", flush=True)

    # -- IVFPQ (production ANN index) ---------------------------------------
    if "ivfpq" in M:
        import math

        nlist = min(4096, max(64, int(8 * math.sqrt(N))))
        m = _divisor_m((out_dim * bits) // 8, dim)
        try:
            t = time.perf_counter()
            index = faiss.index_factory(
                dim, f"IVF{nlist},PQ{m}", faiss.METRIC_INNER_PRODUCT
            )
            index.train(train)
            index.add(C)
            index.nprobe = min(64, nlist)
            bt = time.perf_counter() - t
            _, nn = index.search(Q, 100)
            _, cand = index.search(Q, kcand)
            rr = _rerank(cand, Q, C)
            q1 = bench(lambda index=index: index.search(Q, 10))
            qr = bench(lambda index=index: index.search(Q, kcand))
            add(
                f"faiss-IVFPQ(m={m},nlist={nlist})",
                m,
                bt,
                q1,
                qr,
                recall(gt, nn, 10),
                recall(gt, rr, 10),
                recall(gt, nn, 100),
                f"nprobe={index.nprobe}; +rerank x{oversample}",
            )
        except Exception as e:  # noqa: BLE001
            print(f"  IVFPQ failed: {e}", flush=True)

    # -- RaBitQ (2024 SOTA, ~1 bit/dim) -------------------------------------
    if "rabitq" in M:
        try:
            t = time.perf_counter()
            index = faiss.index_factory(dim, "RaBitQ", faiss.METRIC_INNER_PRODUCT)
            index.train(train)
            index.add(C)
            bt = time.perf_counter() - t
            _, nn = index.search(Q, 100)
            _, cand = index.search(Q, kcand)
            rr = _rerank(cand, Q, C)
            q1 = bench(lambda index=index: index.search(Q, 10))
            qr = bench(lambda index=index: index.search(Q, kcand))
            add(
                "faiss-RaBitQ",
                dim / 8.0,
                bt,
                q1,
                qr,
                recall(gt, nn, 10),
                recall(gt, rr, 10),
                recall(gt, nn, 100),
                f"1-bit/dim; +rerank x{oversample}",
            )
        except Exception as e:  # noqa: BLE001
            print(f"  RaBitQ unavailable in this faiss build: {e}", flush=True)

    # -- PCA-only (truncation, fp32 kept dims) — isolates dimension reduction
    if "pca" in M:
        pca = PCAMatryoshka(input_dim=dim, output_dim=out_dim)
        pca.fit(train)
        t = time.perf_counter()
        Cp = normalize(np.asarray(pca.transform(C), dtype=np.float32))
        Qp = normalize(np.asarray(pca.transform(Q), dtype=np.float32))
        idx = faiss.IndexFlatIP(out_dim)
        idx.add(Cp)
        bt = time.perf_counter() - t
        _, nn = idx.search(Qp, 100)
        _, cand = idx.search(Qp, kcand)
        rr = _rerank(cand, Q, C)
        q1 = bench(lambda: idx.search(Qp, 10))
        qr = bench(lambda: idx.search(Qp, kcand))
        ev = float(np.sum(pca._eigenvalues) / np.sum(pca._all_eigenvalues))
        add(
            f"PCA-only({out_dim}d, fp32)",
            out_dim * 4,
            bt,
            q1,
            qr,
            recall(gt, nn, 10),
            recall(gt, rr, 10),
            recall(gt, nn, 100),
            f"truncation only; var={ev:.2f}; +rerank x{oversample}",
        )

    # -- TQ-only (scalar quant on full dim, no truncation) — isolates SQ ----
    if "tq" in M:
        pcaf = PCAMatryoshka(input_dim=dim, output_dim=dim)
        pcaf.fit(train)
        pipe = pcaf.with_quantizer(bits=bits)
        t = time.perf_counter()
        codes = pipe.compress_batch(C)
        recon = normalize(np.asarray(pipe.decompress_batch(codes), dtype=np.float32))
        idx = faiss.IndexFlatIP(dim)
        idx.add(recon)
        bt = time.perf_counter() - t
        _, nn = idx.search(Q, 100)
        _, cand = idx.search(Q, kcand)
        rr = _rerank(cand, Q, C)
        q1 = bench(lambda: idx.search(Q, 10))
        qr = bench(lambda: idx.search(Q, kcand))
        add(
            f"TQ-only({dim}d, {bits}b)",
            dim * bits / 8.0,
            bt,
            q1,
            qr,
            recall(gt, nn, 10),
            recall(gt, rr, 10),
            recall(gt, nn, 100),
            f"scalar-quant only; +rerank x{oversample}",
        )

    # -- PCA + TQ (the combined pipeline) -----------------------------------
    if "pca_tq" in M:
        pca = PCAMatryoshka(input_dim=dim, output_dim=out_dim)
        pca.fit(train)
        pipe = pca.with_quantizer(bits=bits)
        t = time.perf_counter()
        codes = pipe.compress_batch(C)
        recon = normalize(np.asarray(pipe.decompress_batch(codes), dtype=np.float32))
        rdim = recon.shape[1]
        Qp = (
            normalize(np.asarray(pca.transform(Q), dtype=np.float32))
            if rdim != dim
            else Q
        )
        idx = faiss.IndexFlatIP(rdim)
        idx.add(recon)
        bt = time.perf_counter() - t
        _, nn = idx.search(Qp, 100)
        _, cand = idx.search(Qp, kcand)
        rr = _rerank(cand, Q, C)
        q1 = bench(lambda: idx.search(Qp, 10))
        qr = bench(lambda: idx.search(Qp, kcand))
        add(
            f"PCA+TQ({out_dim}d, {bits}b)",
            out_dim * bits / 8.0,
            bt,
            q1,
            qr,
            recall(gt, nn, 10),
            recall(gt, rr, 10),
            recall(gt, nn, 100),
            f"combined pipeline; +rerank x{oversample}",
        )

    # -- ADCIndex (compressed-domain search, no reconstruction) -------------
    if "adc" in M:
        pca = PCAMatryoshka(input_dim=dim, output_dim=out_dim)
        pca.fit(train)
        t = time.perf_counter()
        index = ADCIndex(pca.with_quantizer(bits=bits)).add(C)
        bt = time.perf_counter() - t
        i1, _ = index.search(Q, k=100)
        ir = index.search(Q, k=10, rerank=oversample, originals=C)
        q1 = bench(lambda: index.search(Q, k=10))
        qr = bench(lambda: index.search(Q, k=10, rerank=oversample, originals=C))
        add(
            f"ADCIndex({out_dim}d, {bits}b)",
            out_dim * bits / 8.0,
            bt,
            q1,
            qr,
            recall(gt, np.asarray(i1), 10),
            recall(gt, np.asarray(ir), 10),
            recall(gt, np.asarray(i1), 100),
            f"compressed-domain ADC; +rerank x{oversample}",
        )

    return rows


# ---------------------------------------------------------------- rendering
COLS = [
    ("method", "Method", "l"),
    ("n", "N", "r"),
    ("dim", "Dim", "r"),
    ("compression_x", "Comp x", "r"),
    ("bytes_per_vec", "B/vec", "r"),
    ("ram_mb", "RAM MB", "r"),
    ("recall_at_10", "R@10", "r"),
    ("recall_at_10_rerank", "R@10 +rr", "r"),
    ("recall_at_100", "R@100", "r"),
    ("qps_1stage", "QPS", "r"),
    ("build_s", "Build s", "r"),
    ("note", "Notes", "l"),
]


def to_markdown(rows: list[dict]) -> str:
    head = "| " + " | ".join(h for _, h, _ in COLS) + " |"
    sep = "|" + "|".join("---:" if a == "r" else "---" for _, _, a in COLS) + "|"
    lines = [head, sep]
    for r in rows:
        cells = []
        for key, _, _ in COLS:
            v = r.get(key)
            cells.append("-" if v is None else str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


## 4. Synthesize an anisotropic, heavy-tailed corpus

In [ ]:
import numpy as np
rng = np.random.default_rng(SEED)
# power-law eigenvalues -> a few dominant directions dominate the variance
eig = (np.arange(1, DIM + 1) ** -ANISO).astype(np.float64)
scales = np.sqrt(eig / eig.sum() * DIM)
# heavy-tailed (Student-t) latent, scaled per-axis, then randomly rotated
def draw(n):
    z = rng.standard_t(TAIL_DF, size=(n, DIM)) * scales
    return z.astype(np.float32)
Qr = np.linalg.qr(rng.standard_normal((DIM, DIM)))[0].astype(np.float32)  # random rotation
C = normalize((draw(N) @ Qr))
Q = normalize((draw(QUERIES) @ Qr))
gt = exact_topk(Q, C, 100)
ev = eig[:OUT_DIM].sum() / eig.sum()
print(f'anisotropy: top-{OUT_DIM}/{DIM} dims retain {ev:.1%} of spectral energy')

## 5. Run the canonical ladder on the pathological data

In [ ]:
rows = run_canonical(C, Q, gt, out_dim=OUT_DIM, bits=BITS,
                     oversample=OVERSAMPLE, threads=THREADS)
import pandas as pd
pd.DataFrame(rows)[['method','compression_x','bytes_per_vec',
    'recall_at_10','recall_at_10_rerank','recall_at_100','note']]

## 6. Read the result
- Because the spectrum is **concentrated** (top-`OUT_DIM` dims hold most energy), PCA truncation *should* stay strong here even at aggressive compression — the mirror image of the GloVe-100 case where the spectrum is flat and truncation hurts.
- Sweep `ANISO` down toward 0 (flatter spectrum) and `TAIL_DF` down (heavier tails) to find where PCA+TQ starts to lose to PQ/OPQ — that boundary is the honest robustness envelope, and it tracks spectral concentration, exactly as `RESULTS_glove.md` argues.
- +rerank recovers recall in every regime (it reranks exact fp32), so the compressed stage's job is candidate recall, not final ranking.